In [0]:
%sql

-- Recebe CEP e numero e cruza ViaCEP com BrasilAPI

CREATE OR REPLACE FUNCTION geocep.default.Fcn_Logradouro(
    p_cep STRING,
    p_numero STRING
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Padroniza enderecos usando ViaCEP e BrasilAPI'
RETURN

WITH parametros AS (
    -- 1. Padronizacao do CEP e numero informados
    SELECT
        REGEXP_REPLACE(p_cep, '[^0-9]', '') AS cep,
        NULLIF(TRIM(p_numero), '') AS numero
),

viacep AS (
    -- 2. Leitura e tratamento da primeira API
    SELECT
        cep,
        CAST(data_consulta AS TIMESTAMP) AS data_consulta,
        NULLIF(TRIM(get_json_object(
            json_original, '$.logradouro')), '') AS logradouro,
        NULLIF(TRIM(get_json_object(
            json_original, '$.bairro')), '') AS bairro,
        NULLIF(TRIM(get_json_object(
            json_original, '$.localidade')), '') AS cidade,
        NULLIF(TRIM(get_json_object(
            json_original, '$.uf')), '') AS uf,

        ROW_NUMBER() OVER (
            PARTITION BY cep
            ORDER BY data_consulta DESC
        ) AS rn

    FROM geocep.default.ref_viacep
    WHERE status = 'OK'
),

brasilapi AS (
    -- 3. Leitura da segunda API e JSON aninhado
    SELECT
        cep,
        CAST(data_consulta AS TIMESTAMP) AS data_consulta,
        NULLIF(TRIM(get_json_object(
            json_original, '$.street')), '') AS logradouro,
        NULLIF(TRIM(get_json_object(
            json_original, '$.neighborhood')), '') AS bairro,
        NULLIF(TRIM(get_json_object(
            json_original, '$.city')), '') AS cidade,
        NULLIF(TRIM(get_json_object(
            json_original, '$.state')), '') AS uf,

        -- Desaninhamento das coordenadas do JSON
        TRY_CAST(get_json_object(
            json_original,
            '$.location.coordinates.latitude'
        ) AS DOUBLE) AS latitude,

        TRY_CAST(get_json_object(
            json_original,
            '$.location.coordinates.longitude'
        ) AS DOUBLE) AS longitude,

        ROW_NUMBER() OVER (
            PARTITION BY cep
            ORDER BY data_consulta DESC
        ) AS rn

    FROM geocep.default.ref_brasilapi
    WHERE status = 'OK'
),

consolidado AS (
    -- 4. Cruzamento das duas fontes pelo CEP
    SELECT
        p.cep,
        p.numero,

        COALESCE(v.logradouro, b.logradouro) AS logradouro,
        COALESCE(v.bairro, b.bairro) AS bairro,
        COALESCE(v.cidade, b.cidade) AS cidade,
        UPPER(COALESCE(v.uf, b.uf)) AS uf,

        b.latitude,
        b.longitude,

        GREATEST(
            v.data_consulta,
            b.data_consulta
        ) AS data_referencia,

        CASE
            WHEN v.cep IS NOT NULL AND b.cep IS NOT NULL
                THEN 'ViaCEP + BrasilAPI'
            WHEN v.cep IS NOT NULL
                THEN 'ViaCEP'
            WHEN b.cep IS NOT NULL
                THEN 'BrasilAPI'
            ELSE 'Nao encontrado'
        END AS fonte,

        CASE
            WHEN v.cep IS NULL AND b.cep IS NULL
                THEN 'NAO_ENCONTRADO'
            WHEN v.cep IS NULL
                THEN 'FALLBACK_BRASILAPI'
            WHEN b.cep IS NULL
                THEN 'SOMENTE_VIACEP'
            WHEN LOWER(v.logradouro) = LOWER(b.logradouro)
                THEN 'VALIDADO'
            ELSE 'DIVERGENCIA'
        END AS status_qualidade

    FROM parametros p

    LEFT JOIN viacep v
        ON p.cep = v.cep AND v.rn = 1

    LEFT JOIN brasilapi b
        ON p.cep = b.cep AND b.rn = 1
),

resultado AS (
    -- 5. Montagem do endereco e metricas de qualidade
    SELECT
        cep,
        numero,
        logradouro,
        bairro,
        cidade,
        uf,
        latitude,
        longitude,
        fonte,
        data_referencia,
        status_qualidade,

        CONCAT_WS(', ',
            CASE
                WHEN logradouro IS NOT NULL
                THEN CONCAT_WS(', ', logradouro, numero)
            END,
            bairro,
            CONCAT_WS(' - ', cidade, uf),
            CASE
                WHEN LENGTH(cep) = 8
                THEN cep
            END
        ) AS endereco_completo,

        -- Percentual de preenchimento dos 6 campos
        ROUND(100.0 * (
            IF(LENGTH(cep) = 8, 1, 0) +
            IF(numero IS NOT NULL, 1, 0) +
            IF(logradouro IS NOT NULL, 1, 0) +
            IF(bairro IS NOT NULL, 1, 0) +
            IF(cidade IS NOT NULL, 1, 0) +
            IF(uf IS NOT NULL, 1, 0)
        ) / 6, 2) AS percentual_completude,

        CASE
            WHEN LENGTH(cep) <> 8 THEN 'CEP_INVALIDO'
            WHEN status_qualidade = 'NAO_ENCONTRADO'
                THEN 'NAO_ENCONTRADO'
            WHEN numero IS NULL THEN 'NUMERO_AUSENTE'
            WHEN logradouro IS NULL OR cidade IS NULL OR uf IS NULL
                THEN 'ENDERECO_INCOMPLETO'
            WHEN status_qualidade = 'DIVERGENCIA'
                THEN 'VERIFICAR_DIVERGENCIA'
            ELSE 'OK'
        END AS status_dq

    FROM consolidado
)

SELECT * FROM resultado;
